In [ ]:
!pip install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 71.1 MB/s eta 0:00:00


In [ ]:
import os
from glob import glob
import mne
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class EEGDataset(Dataset):
    """
    PyTorch Dataset for loading EEG data from EDF files using MNE.

    Args:
        data_dir (str): Path to directory containing .edf files.
        channels (list of str): EEG channel names to load. If None, load all.
        preload (bool): Whether to preload data into memory.
        transform (callable, optional): Optional transform to apply to each sample.
    """
    def __init__(self, data_dir, channels=None, preload=True, transform=None):
        self.files = sorted(glob(os.path.join(data_dir, '*.edf')))
        self.channels = channels
        self.preload = preload
        self.transform = transform
        self._raws = []

        if self.preload:
            for fpath in self.files:
                raw = mne.io.read_raw_edf(fpath, preload=True, verbose=False)
                if self.channels:
                    raw.pick_channels(self.channels)
                self._raws.append(raw)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        # Load or retrieve raw data
        if self.preload:
            raw = self._raws[idx]
        else:
            fpath = self.files[idx]
            raw = mne.io.read_raw_edf(fpath, preload=True, verbose=False)
            if self.channels:
                raw.pick_channels(self.channels)

        # get data as numpy array: shape (n_channels, n_times)
        data, times = raw.get_data(return_times=True)
        eeg_tensor = torch.from_numpy(data).float()
        times_tensor = torch.from_numpy(times).float()

        sample = {
            'eeg': eeg_tensor,
            'times': times_tensor,
        }

        if self.transform:
            sample = self.transform(sample)

        return sample


def pad_collate(batch):
    """
    Collate function to pad EEG tensors in a batch along the time dimension.

    Args:
        batch (list of dicts): Each dict has keys 'eeg', 'times'.
    Returns:
        dict with batched 'eeg' and 'times' tensors.
    """
    max_len = max(sample['eeg'].shape[1] for sample in batch)

    eegs, times = [], []
    for sample in batch:
        eeg, t = sample['eeg'], sample['times']
        pad_len = max_len - eeg.shape[1]
        if pad_len > 0:
            eeg = F.pad(eeg, (0, pad_len))
            t = F.pad(t, (0, pad_len))
        eegs.append(eeg)
        times.append(t)

    eeg_batch = torch.stack(eegs)   # (batch_size, channels, max_time)
    times_batch = torch.stack(times)  # (batch_size, max_time)

    return {'eeg': eeg_batch, 'times': times_batch}

# Example usage
data_dir = '/content/drive/MyDrive/nutsh/hackathons/EF Builders Retreat + Cambridge MIND/data/'
channels = None

dataset = EEGDataset(data_dir, channels=channels, preload=True)
dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True,
    num_workers=2,
    pin_memory=False,
    collate_fn=pad_collate,
    prefetch_factor=1,
    persistent_workers=False
)

for batch in dataloader:
    eeg = batch['eeg']       # (batch, channels, max_time)
    times = batch['times']   # (batch, max_time)
    print(eeg.shape, times.shape)
    break


<ipython-input-1-05c20eef0a86>:27: RuntimeWarning: Physical range is not defined in following channels:
sams_valence, sams_arousal, sams_valencert, sams_arousalrt, nback_stimuli, nback_keypress
  raw = mne.io.read_raw_edf(fpath, preload=True, verbose=False)
<ipython-input-1-05c20eef0a86>:27: RuntimeWarning: Physical range is not defined in following channels:
sams_valence, sams_arousal, sams_valencert, sams_arousalrt, nback_stimuli, nback_keypress
  raw = mne.io.read_raw_edf(fpath, preload=True, verbose=False)


torch.Size([1, 47, 742000]) torch.Size([1, 742000])


In [ ]:
# Fetch first batch
batch = next(iter(dataloader))
eeg_batch = batch['eeg']    # shape: (B, n_ch, T)
times_batch = batch['times']  # shape: (B, T)
print(f"Batch EEG shape: {eeg_batch.shape}")

# Plot topomap for first timestamp of first sample
# Extract data vector
data_vector = eeg_batch[0, :, 0].numpy()  # values at t=0

# Use the first raw file's info to get channel positions
raw0 = dataset._raws[0]
# Ensure standard montage is set
try:
    raw0.set_montage('standard_1020')
except Exception:
    mont = mne.channels.make_standard_montage('standard_1020')
    raw0.set_montage(mont)
info = raw0.info

# Plot
fig, ax = plt.subplots()
mne.viz.plot_topomap(data_vector, info, axes=ax, show=False)
ax.set_title('EEG Topomap at First Timestamp')
plt.show()

Batch EEG shape: torch.Size([1, 47, 742000])


ValueError: DigMontage is only a subset of info. There are 15 channel positions not present in the DigMontage. The channels missing from the montage are:

['ECG', 'ft_valance', 'ft_arousal', 'ft_x', 'ft_y', 'ft_ghostvalence', 'ft_ghostarousal', 'music', 'trialtype', 'sams_valence', 'sams_arousal', 'sams_valencert', 'sams_arousalrt', 'nback_stimuli', 'nback_keypress'].

Consider using inst.rename_channels to match the montage nomenclature, or inst.set_channel_types if these are not EEG channels, or use the on_missing parameter if the channel positions are allowed to be unknown in your analyses.

In [ ]:
# prompt: plot the first channel of the first batch

import matplotlib.pyplot as plt

# Assuming 'eeg' is the tensor from the previous code block
first_channel = eeg[0, 0, :]  # First batch, first channel
time_points = times[0, :]

plt.plot(time_points, first_channel)
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.title('First Channel of the First Batch')
plt.show()

In [ ]:
!git clone https://github.com/state-spaces/s4.git

fatal: destination path 's4' already exists and is not an empty directory.


In [ ]:
%cd s4
!pip install -r requirements.txt

/content/s4
DEPRECATION: Loading egg at /usr/local/lib/python3.11/dist-packages/structured_kernels-0.1.0-py3.11-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat

from src.models.nn import DropoutNd

class S4DKernel(nn.Module):
    """Generate convolution kernel from diagonal SSM parameters, and optionally return latent state evolution."""

    def __init__(self, d_model, N=64, dt_min=0.001, dt_max=0.1, lr=None):
        super().__init__()
        H = d_model
        log_dt = torch.rand(H) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)

        # C = torch.randn(H, N // 2, dtype=torch.cfloat)
        # self.C = nn.Parameter(torch.view_as_real(C))
        self.register("log_dt", log_dt, lr)

        log_A_real = torch.log(0.5 * torch.ones(H, N // 2))
        A_imag = math.pi * repeat(torch.arange(N // 2), 'n -> h n', h=H)
        self.register("log_A_real", log_A_real, lr)
        self.register("A_imag", A_imag, lr)

    def forward(self, L):
        """
        returns: (H, L) convolution kernel over length L
        """
        dt = torch.exp(self.log_dt)               # (H)
        C = torch.view_as_complex(self.C)         # (H, N/2)
        A = -torch.exp(self.log_A_real) + 1j * self.A_imag  # (H, N/2)

        dtA = A * dt.unsqueeze(-1)               # (H, N/2)
        C_mod = C * (torch.exp(dtA) - 1.0) / A   # (H, N/2)

        t = torch.arange(L, device=A.device)     # (L)
        exp_term = torch.exp(dtA.unsqueeze(-1) * t)  # (H, N/2, L)

        K = 2 * torch.einsum('hn,hnl->hl', C_mod, exp_term).real  # (H, L)
        return K

    def latent_forward(self, L):
        """
        returns: (H, N, L) concatenated real and imag latent states for each eigenmode over length L
        """
        dt = torch.exp(self.log_dt)               # (H)
        A = -torch.exp(self.log_A_real) + 1j * self.A_imag  # (H, N/2)
        dtA = A * dt.unsqueeze(-1)               # (H, N/2)

        t = torch.arange(L, device=A.device)     # (L)
        X = torch.exp(dtA.unsqueeze(-1) * t)     # (H, N/2, L)

        X_real = X.real                          # (H, N/2, L)
        X_imag = X.imag                          # (H, N/2, L)
        X_concat = torch.cat([X_real, X_imag], dim=1)  # (H, N, L)
        return X_concat

    def register(self, name, tensor, lr=None):
        """Register a tensor with a configurable learning rate and 0 weight decay"""
        if lr == 0.0:
            self.register_buffer(name, tensor)
        else:
            self.register_parameter(name, nn.Parameter(tensor))
            optim = {"weight_decay": 0.0}
            if lr is not None:
                optim["lr"] = lr
            setattr(getattr(self, name), "_optim", optim)

class S4D(nn.Module):
    def __init__(self, d_model, d_state=64, dropout=0.0, transposed=True, **kernel_args):
        super().__init__()
        self.h = d_model
        self.n = d_state
        self.transposed = transposed
        self.kernel = S4DKernel(self.h, N=self.n, **kernel_args)
        self.activation = nn.GELU()
        dropout_fn = DropoutNd
        self.dropout = dropout_fn(dropout) if dropout > 0.0 else nn.Identity()

    def forward(self, u, **kwargs):
        """ Input and output shape (B, H, L) """
        if not self.transposed:
            u = u.transpose(-1, -2)
        L = u.size(-1)

        k = self.kernel(L=L)                  # (H, L)
        k_f = torch.fft.rfft(k, n=2*L)       # (H, L)
        u_f = torch.fft.rfft(u, n=2*L)       # (B, H, L)
        y = torch.fft.irfft(u_f * k_f, n=2*L)[..., :L]  # (B, H, L)

        y = self.dropout(self.activation(y))
        if not self.transposed:
            y = y.transpose(-1, -2)
        return y, None

    def latent_forward(self, u, **kwargs):
        """Compute latent state trajectories modulated by input u. Returns (B, H, N, L) """
        if not self.transposed:
            u = u.transpose(-1, -2)
        B, H, L = u.shape

        X = self.kernel.latent_forward(L)     # (H, N, L)
        u_exp = u.unsqueeze(2)                # (B, H, 1, L)
        X_batched = X.unsqueeze(0)            # (1, H, N, L)
        X_resp = u_exp * X_batched            # (B, H, N, L)

        if not self.transposed:
            X_resp = X_resp.transpose(2, 3)
        return X_resp, None

# -----------------------
# Sparse Autoencoder Setup
# -----------------------
class S4DAutoencoder(nn.Module):
    def __init__(self, d_model, d_state=64, n_points=10, dropout=0.0, transposed=True, decoder_channels=None, **kernel_args):
        super().__init__()
        self.n_points = n_points
        self.encoder = S4D(d_model, d_state, dropout, transposed, **kernel_args)
        # Decoder: 1x1 conv to reconstruct only last n_points
        self.decoder = nn.Conv1d(in_channels=d_model * d_state * 2, out_channels=d_model, kernel_size=1)

    def forward(self, x):
        # x: (B, H, L)
        # Obtain latent states for entire sequence
        print("x",x.shape)
        latent, _ = self.encoder.latent_forward(x)       # (B, H, N, L)
        B, H, N, L = latent.shape
        # Flatten latent dims
        print(latent.shape)
        latent_flat = latent.view(B, H * N, L)          # (B, H*N, L)
        # Decode full sequence
        print(latent_flat.shape)
        recon_full = self.decoder(latent_flat)          # (B, H, L)
        # Only keep last n_points
        recon_last = recon_full[..., -self.n_points:]   # (B, H, n_points)
        # Also get target slice
        target_last = x[..., -self.n_points:]           # (B, H, n_points)
        return recon_last, target_last, latent_flat

# -----------------------
# Training Loop
# -----------------------
def train_autoencoder(model, train_loader, epochs=10, lr=1e-3, sparsity_weight=1e-4,
                      use_kl=False, rho=0.05, device='cuda'):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for x in train_loader:
            x = x['eeg'].to(device)                            # (B, H, L)
            recon, target, latent = model(x)            # recon/target: (B,H,n_points)
            # Reconstruction loss on last n_points
            mse = F.mse_loss(recon, target)
            # Sparsity penalty on latent
            if use_kl:
                mean_act = torch.mean(torch.sigmoid(latent), dim=(0, 2))
                kl = torch.sum(rho * torch.log(rho / (mean_act + 1e-8)) +
                               (1 - rho) * torch.log((1 - rho) / (1 - mean_act + 1e-8)))
                sparsity_penalty = kl
            else:
                sparsity_penalty = torch.mean(torch.abs(latent))
            loss = mse + sparsity_weight * sparsity_penalty
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
        avg_loss = total_loss / len(train_loader.dataset)
        print(f"Epoch {epoch}/{epochs} - Loss: {avg_loss:.6f}")

f_fmri = 0.5
f_eeg = 1000

# Example usage:
model = S4DAutoencoder(d_model=47, d_state=2, n_points=f_fmri//f_eeg)
train_autoencoder(model, dataloader, epochs=20, use_kl=True)

ModuleNotFoundError: No module named 'src'

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Import your existing functions
from drive.MyDrive.nutsh.hackathons.read_fMRI import align_eeg_and_fmri_sub02, segment_trials_sub02

class EEGFMRIDataset(Dataset):
    def __init__(self,
                 subjects,
                 runs,
                 base_dir,
                 trial_duration_s=40.0,
                 ttl_code=768,
                 transform=None):
        """
        subjects: list of subject IDs (e.g. ['02', '03', ...])
        runs:     list of run names (e.g. ['genMusic01', 'genMusic02', ...])
        base_dir: root folder containing sub-XX directories
        """
        self.transform = transform
        self.data = []  # to store (eeg_trial, fmri_trial) tuples

        for sub in subjects:
            sub_dir = os.path.join(base_dir)
            for run in runs:
                # 1) align EEG↔fMRI
                eeg, sfreq, fmri_4d, TR = align_eeg_and_fmri_sub02(
                    run,
                    base=sub_dir
                )
                # 2) segment into trials
                events_tsv = os.path.join(
                    sub_dir,
                    'eeg',
                    f"sub-{sub}_task-{run}_events.tsv"
                )
                trials_eeg, trials_fmri = segment_trials_sub02(
                    eeg, sfreq,
                    fmri_4d, TR,
                    events_tsv,
                    trial_duration_s=trial_duration_s,
                    ttl_code=ttl_code
                )

                # 3) accumulate
                n_trials = trials_eeg.shape[0]
                for i in range(n_trials):
                    eeg_trial = trials_eeg[i]            # shape (n_chan, samples)
                    fmri_trial = trials_fmri[i]          # shape (X, Y, Z, vols)
                    self.data.append((eeg_trial, fmri_trial))

        print(f"Total trials across all subjects/runs: {len(self.data)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        eeg_np, fmri_np = self.data[idx]

        # convert to torch.Tensor, permute fmri to (vols, X, Y, Z)
        eeg_tensor  = torch.from_numpy(eeg_np).float()                           # (n_chan, samples)
        fmri_tensor = torch.from_numpy(fmri_np).float()                          # (X, Y, Z, samples)

        if self.transform:
            eeg_tensor, fmri_tensor = self.transform(eeg_tensor, fmri_tensor)

        return eeg_tensor, fmri_tensor

import torch
import torch.nn.functional as F

def collate_fn(batch):
    """
    batch: list of tuples (eeg, fmri), where
      - eeg: Tensor (n_channels, n_samples_eeg)
      - fmri: Tensor (n_vols, X, Y, Z)
    Returns:
      - eeg_resampled: Tensor (B, n_channels, n_vols)
      - fmri_batch:    Tensor (B, n_vols, X, Y, Z)
    """
    # unzip
    eeg_list, fmri_list = zip(*batch)

    # stack into (B, C, L_eeg) and (B, V, X, Y, Z)
    eeg_batch  = torch.stack(eeg_list, dim=0)   # (B, n_chan, n_samples_eeg)
    fmri_batch = torch.stack(fmri_list, dim=0)  # (B, n_vols, X, Y, Z)

    # number of volumes to match
    n_vols = fmri_batch.shape[-1]

    # resample EEG from L_eeg → n_vols
    # F.interpolate works on 3D inputs for 1D interpolation: (B, C, L)
    eeg_resampled = F.interpolate(
        eeg_batch,
        size=n_vols,
        mode='linear',
        align_corners=False
    )  # (B, n_chan, n_vols)

    # 4) flatten fMRI spatial dims into "channels"
    B, X, Y, Z, V = fmri_batch.shape
    chans = X * Y * Z
    fmri_flat = fmri_batch.reshape(B, chans, V)  # → (B, n_vols, chans)

    return eeg_resampled, fmri_flat

# ----------------------------
# Usage example
# ----------------------------
# specify your subjects and runs
subjects = [f"{i:02d}" for i in [2]]
runs     = ['genMusic01']#,'genMusic02','genMusic03']
base_dir = '/content/drive/MyDrive/nutsh/hackathons/EF Builders Retreat + Cambridge MIND/data'

dataset = EEGFMRIDataset(
    subjects=subjects,
    runs=runs,
    base_dir=base_dir,
    trial_duration_s=40.0,
    ttl_code=768
)

# create DataLoader
loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True,
    num_workers=4,      # adjust for your system
    pin_memory=True,     # if using GPU
    collate_fn=collate_fn
)

# iterate
for batch_idx, (eeg_batch, fmri_batch) in enumerate(loader):
    # eeg_batch: (B, n_chan, samples)
    # fmri_batch: (B, vols, X, Y, Z)
    print(f"Batch {batch_idx}:")
    print(" EEG:", eeg_batch.shape)
    print(" fMRI:", fmri_batch.shape)
    # ... your training/analysis code ...
    break


=== Aligning run: genMusic01 ===
Raw EEG: 47 channels × 600000 samples
 After pick(): 46 EEG channels × 600000 samples
 First TTL onset (s): 35.683
 TTL sample index: 35683
 After trim: 46 channels × 564317 samples
 Loaded fMRI data shape: (64, 64, 37, 274) (X×Y×Z×n_vols)
 fMRI vols: 274    TR(s): 2.0
 Needed EEG samples: 548000
 After CROP: 548000 samples (cropped)
✔ Final aligned EEG shape: 46 channels × 548000 samples
✔ Final fMRI    shape: (64, 64, 37, 274)
Found 24 trial starts at:
 [ 37.307  37.354  83.812  83.816 129.869 129.874 174.794 174.798 221.503
 221.508 265.843 265.847 309.23  309.234 352.802 352.807 396.857 396.861
 443.232 443.236 489.29  489.294 533.513 533.517]
Each trial → 40000 EEG samples  |  20 fMRI vols
  • skipping EEG at 533.513s → only 14487 samples
  • skipping EEG at 533.517s → only 14483 samples
✔ Successfully segmented:
  • trials_eeg shape : (22, 46, 40000) (n_trials, n_chan, samples)
  • trials_fmri shape: (22, 64, 64, 37, 20) (n_trials, X, Y, Z, vols)

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Batch 0:
 EEG: torch.Size([1, 46, 20])
 fMRI: torch.Size([1, 151552, 20])


In [ ]:
class EEGEncoder(nn.Module):                 # [B, 46] ➜ [B, d]
    def __init__(self, d=128):
        super().__init__()
        self.proj = nn.Linear(46, d)

    def forward(self, x):
        return F.normalize(self.proj(x), p=2, dim=-1)   # L2-norm

class FMRIEncoder(nn.Module):                # [B, 151 552] ➜ [B, d]
    def __init__(self, d=128):
        super().__init__()
        self.proj = nn.Linear(151_552, d)

    def forward(self, x):
        return F.normalize(self.proj(x), p=2, dim=-1)

class TwoTower(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.eeg  = EEGEncoder(d)
        self.fmri = FMRIEncoder(d)

    def forward(self, eeg, fmri):
        return self.eeg(eeg), self.fmri(fmri)   # two [B, d] tensors

def cross_modal_nce(z_eeg, z_fmri, *, temperature=0.07):
    """
    z_eeg   : (B, d)
    z_fmri  : (B, d)
    Positive pair i is (z_eeg[i], z_fmri[i]).
    All cross-modal mismatches are negatives.
    """
    logits  = z_eeg @ z_fmri.T / temperature        # (B, B)
    targets = torch.arange(len(z_eeg), device=z_eeg.device)

    # minimise −log p(positive)
    loss_a2b = F.cross_entropy(logits, targets)     # EEG➜fMRI
    loss_b2a = F.cross_entropy(logits.T, targets)   # fMRI➜EEG
    return 0.5 * (loss_a2b + loss_b2a)

In [ ]:
class TimeSlicePairs(Dataset):
    """Takes a (C, T) tensor and returns T examples of shape (C,)."""
    def __init__(self, eeg_3d, fmri_3d):
        # eeg_3d  : [C_eeg,  T]
        # fmri_3d : [C_fmri, T]
        self.eeg  = eeg_3d.permute(1, 0).contiguous()   # (T, C_eeg)
        self.fmri = fmri_3d.permute(1, 0).contiguous()  # (T, C_fmri)
        assert len(self.eeg) == len(self.fmri)

    def __len__(self):
        return len(self.eeg)        # = T

    def __getitem__(self, idx):
        return self.eeg[idx], self.fmri[idx]

# -------- usage --------
dataset  = TimeSlicePairs(eeg_raw, fmri_raw)
loader   = DataLoader(dataset, batch_size=8, shuffle=True)


In [ ]:
# ------------------------------------------------------------
# 0.  Imports & DATASET exactly as you supplied
# ------------------------------------------------------------
import os, torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# ------------------------------------------------------------------
# Put your existing helper imports here (alignment, segmentation …)
# from drive.MyDrive.nutsh.hackathons.read_fMRI import ...
# ------------------------------------------------------------------

def collate_fn(batch):
    """
    *Unchanged* from your message except the final comment row.
    Returns:
        eeg_resampled : (B, C_eeg, T)
        fmri_flat     : (B, C_fmri, T)   where C_fmri = X·Y·Z
    """
    eeg_list, fmri_list = zip(*batch)
    eeg_batch  = torch.stack(eeg_list,  dim=0)       # (B, C_eeg, L_eeg)
    fmri_batch = torch.stack(fmri_list, dim=0)       # (B, X, Y, Z, T)

    n_vols = fmri_batch.shape[-1]

    eeg_resampled = F.interpolate(
        eeg_batch, size=n_vols, mode='linear', align_corners=False
    )                                               # (B, C_eeg, T)

    B, X, Y, Z, T = fmri_batch.shape
    fmri_flat = fmri_batch.reshape(B, X*Y*Z, T)     # (B, C_fmri, T)
    return eeg_resampled, fmri_flat


# ------------------------------------------------------------
# 1.  Two-tower model
# ------------------------------------------------------------
class TwoTower(nn.Module):
    def __init__(self, eeg_dim, fmri_dim, latent_dim=128):
        super().__init__()
        self.eeg_proj  = nn.Linear(eeg_dim,  latent_dim)
        self.fmri_proj = nn.Linear(fmri_dim, latent_dim)

    def forward(self, eeg_vecs, fmri_vecs):
        z_eeg  = F.normalize(self.eeg_proj(eeg_vecs),  p=2, dim=-1)
        z_fmri = F.normalize(self.fmri_proj(fmri_vecs), p=2, dim=-1)
        return z_eeg, z_fmri


# ------------------------------------------------------------
# 2.  Cross-modal NT-Xent loss
# ------------------------------------------------------------
def cross_modal_nce(z_a, z_b, *, temperature=0.07):
    """
    z_a, z_b : (N, d) — embeddings from the two modalities.
    Diagonal pairs are positives, off-diagonals are negatives.
    """
    logits  = (z_a @ z_b.T) / temperature           # (N, N)
    targets = torch.arange(len(z_a), device=z_a.device)
    loss_a2b = F.cross_entropy(logits,   targets)   # EEG ➜ fMRI
    loss_b2a = F.cross_entropy(logits.T, targets)   # fMRI ➜ EEG
    return 0.5 * (loss_a2b + loss_b2a)


# ------------------------------------------------------------
# 3.  Instantiate DATASET  ➜  DATALOADER
# ------------------------------------------------------------
subjects = [f"{i:02d}" for i in [2]]
runs     = ["genMusic01"]
base_dir = "/content/drive/MyDrive/nutsh/hackathons/EF Builders Retreat + Cambridge MIND/data"

dataset = EEGFMRIDataset(
    subjects=subjects,
    runs=runs,
    base_dir=base_dir,
    trial_duration_s=40.0,
    ttl_code=768
)

loader = DataLoader(
    dataset,
    batch_size=1,          # one *trial* per worker; time-points explode later
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=collate_fn,
)


# ------------------------------------------------------------
# 4.  Training loop
# ------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# -- peek one batch to get feature dims --
with torch.no_grad():
    eeg_sample, fmri_sample = next(iter(loader))    # shapes (1, C_eeg, T) / (1, C_fmri, T)
    C_eeg   = eeg_sample.shape[1]
    C_fmri  = fmri_sample.shape[1]

model = TwoTower(C_eeg, C_fmri, latent_dim=128).to(device)
optim = torch.optim.AdamW(model.parameters(), lr=1e-1)

for epoch in range(30):
    running_loss = 0.0
    for eeg, fmri in loader:
        # eeg, fmri : (B, C, T)
        eeg  = eeg.to(device, non_blocking=True)
        fmri = fmri.to(device, non_blocking=True)

        # -------- explode time axis into examples --------
        # (B, C, T)  →  (B*T, C)
        B, C_eeg, T = eeg.shape
        _, C_fmri, _ = fmri.shape
        eeg_vecs  = eeg.permute(0, 2, 1).reshape(-1, C_eeg)    # (B*T, C_eeg)
        fmri_vecs = fmri.permute(0, 2, 1).reshape(-1, C_fmri)  # (B*T, C_fmri)

        # -------- forward & loss --------
        z_eeg, z_fmri = model(eeg_vecs, fmri_vecs)
        loss = cross_modal_nce(z_eeg, z_fmri)

        optim.zero_grad()
        loss.backward()
        optim.step()

        running_loss += loss.item()

    print(f"Epoch {epoch:02d} | NT-Xent loss = {running_loss/len(loader):.4f}")


=== Aligning run: genMusic01 ===
Raw EEG: 47 channels × 600000 samples
 After pick(): 46 EEG channels × 600000 samples
 First TTL onset (s): 35.683
 TTL sample index: 35683
 After trim: 46 channels × 564317 samples
 Loaded fMRI data shape: (64, 64, 37, 274) (X×Y×Z×n_vols)
 fMRI vols: 274    TR(s): 2.0
 Needed EEG samples: 548000
 After CROP: 548000 samples (cropped)
✔ Final aligned EEG shape: 46 channels × 548000 samples
✔ Final fMRI    shape: (64, 64, 37, 274)
Found 24 trial starts at:
 [ 37.307  37.354  83.812  83.816 129.869 129.874 174.794 174.798 221.503
 221.508 265.843 265.847 309.23  309.234 352.802 352.807 396.857 396.861
 443.232 443.236 489.29  489.294 533.513 533.517]
Each trial → 40000 EEG samples  |  20 fMRI vols
  • skipping EEG at 533.513s → only 14487 samples
  • skipping EEG at 533.517s → only 14483 samples
✔ Successfully segmented:
  • trials_eeg shape : (22, 46, 40000) (n_trials, n_chan, samples)
  • trials_fmri shape: (22, 64, 64, 37, 20) (n_trials, X, Y, Z, vols)

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch 00 | NT-Xent loss = 2.9964
Epoch 01 | NT-Xent loss = 2.9958
Epoch 02 | NT-Xent loss = 2.9957
Epoch 03 | NT-Xent loss = 2.9957
Epoch 04 | NT-Xent loss = 2.9957
Epoch 05 | NT-Xent loss = 2.9957
Epoch 06 | NT-Xent loss = 2.9956
Epoch 07 | NT-Xent loss = 2.9956
Epoch 08 | NT-Xent loss = 2.9956
Epoch 09 | NT-Xent loss = 2.9954
Epoch 10 | NT-Xent loss = 2.9981
Epoch 11 | NT-Xent loss = 2.9957
Epoch 12 | NT-Xent loss = 2.9957
Epoch 13 | NT-Xent loss = 2.9957
Epoch 14 | NT-Xent loss = 2.9956
Epoch 15 | NT-Xent loss = 2.9956
Epoch 16 | NT-Xent loss = 2.9956
Epoch 17 | NT-Xent loss = 2.9956
Epoch 18 | NT-Xent loss = 2.9956
Epoch 19 | NT-Xent loss = 2.9956
Epoch 20 | NT-Xent loss = 2.9956
Epoch 21 | NT-Xent loss = 2.9956
Epoch 22 | NT-Xent loss = 2.9956
Epoch 23 | NT-Xent loss = 2.9956
Epoch 24 | NT-Xent loss = 2.9955
Epoch 25 | NT-Xent loss = 2.9955
Epoch 26 | NT-Xent loss = 2.9954
Epoch 27 | NT-Xent loss = 2.9951
Epoch 28 | NT-Xent loss = 2.9960
Epoch 29 | NT-Xent loss = 2.9957
